# ARC3 actual teacher-adaptation pilot

Strict7200-second total development session. BF16 frozen original27B with rank16 supported language adapters. Real observed public teacher data, whole-game gradient holdout. This is a research candidate; CPU gates/loss are not gameplay success. No official submission. Qwen, Hugging Face, FLA and original agent authors retain attribution/licenses.

In [ ]:
import time,json,os,sys,subprocess,signal
from pathlib import Path
NOTEBOOK_START_TIME=time.time()
WORKING_DIR=Path('/kaggle/working')
(WORKING_DIR/'hydra_training_provenance.json').write_text(json.dumps({'candidate': 'qwen27-language-lora-pilot', 'official_submission_eligible': False, 'hard_session_timeout_seconds': 7200, 'internal_total_seconds': 7200, 'training_reserve_seconds': 900, 'model': 'dangkhoa2016/qwen-qwen3-8-27b/Transformers/default/1', 'processor': 'mikedan7/qwen3-8-27b-fp8-official/PyTorch/hf-fp8/1', 'cpu_data_gate': 'kunalsinghh25/arc-hydra-training-data-offline-probe/2', 'cpu_report_sha256': 'b88e44d87bbbb8ba2f68580ea509605a955c9b3c2f5a272348eb79eb0c7c6b45', 'input_archive_sha256': '10e5df7fbe82f82f4cbf95e923e465b900184e36fa9877fbafdf752e6b549961', 'trainer_sha256': 'fb049ff129d2960d66d37d06bfeb69f2bb668c1e987bd3373fe7fc3ee986ad3f', 'exporter_sha256': 'dc9f813ed55cd7f7dbfa5114bc6859be9a96111264c3425f2ce749640f58dcdd', 'runtime_gdn_normalizer_sha256': '4ca4271197f4636515bbac30636ac5187c19396a12e2653a0b04d07f23cedbeb', 'initial_plan': 'At most one epoch of158 actual teacher examples at16K, rank16 on400 supported language attention/recurrent/feed-forward projections,108789760 adapter parameters, LR1e-5, accumulation4. Save after first update and every8updates. Stop if no update by40minutes. Actual CUDA loss/gradients/optimizer, finite nonzero adapter and independent-slot save/reload checks. No serving/gameplay/official-score claim.', 'cpu_placement_boundary': 'CPU native inventory covers all400 selected projections; CPU PEFT placement checked64 attention projections only. Actual400-module placement and shapes must be checked on the GPU before training.', 'deployment_boundary': 'Exact runtime source supports the selected packed LoRA families and DFlash. Enabling LoRA disables a fused GDN projection optimization. Actual saved adapter serving, correct request routing and performance are still required; this training notebook cannot be submitted.'},indent=2))
MODEL_DIR='/kaggle/input/models/dangkhoa2016/qwen-qwen3-8-27b/transformers/default/1'
WHEELHOUSE_DIR='/kaggle/input/datasets/dfranzen/pennyroyal-v253'
SERVED_MODEL_NAME='training-only'
SERVED_MODEL_PORT=8001
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN','').lower() in ['1','true']:raise RuntimeError('Development training only')
# Paste this entire file into the launcher cell.
# Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
# WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
# MODEL_DIR: Intel AutoRound INT4 target (BF16 PLE or albucino FP8 PLE).
# DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
import copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path

PREFIX = "/tmp/sgl-intel"       # Separate from the previous fork's /tmp/sgl venv.
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=(116+12+8)*1024,
    MEMFRAC=0.92,
    MAXREQ=4,
    CUDAGRAPH_MAXBS=4,
    MAMBA_CACHE=24,
    CHUNK=2048,
    MAX_PREFILL=2048,
    SPEC=True,
    FRSPEC=False,
    AUTOTUNE=True,
    # Preserve the original Kaggle checkpoint prefetch/cache behavior.
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="triton",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="float32",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


# ---- validate notebook inputs / identify the offline bundle ----
MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
# Refuse to start a second server on an occupied port; do not kill other jobs.
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
Path(PREFIX).mkdir(parents=True, exist_ok=True)
Path(LOG).parent.mkdir(parents=True, exist_ok=True)
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
if not (libdir / "libcuda.so").exists():
    candidates = []
    for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
    if not candidates:
        code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
    if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
    link = libdir / "libcuda.so"
    if link.is_symlink(): link.unlink()
    link.symlink_to(candidates[0].resolve())
CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import pathlib, sglang, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA unavailable"
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)



uv=[str(Path(PREFIX)/'bin/uv')]
ARCHIVE_SHA256='10e5df7fbe82f82f4cbf95e923e465b900184e36fa9877fbafdf752e6b549961'
EXPECTED_INPUT_MANIFEST={'files': {'16k/train.jsonl': {'sha256': 'fd73b3802e886bd4dc6c443ac24388f3c491f7720b6213ded89e4bc5889d71c5', 'bytes': 7031316}, '16k/validation.jsonl': {'sha256': 'c8527dde162787d496e842a6a4684cf692e9490211b8ce520176ade02c8ea452', 'bytes': 2815735}, '32k/train.jsonl': {'sha256': 'd3e8c271f7fc67c3c5ff72019946ff5a46bf4f90f0bbc973b08da58c44c19efe', 'bytes': 18380809}, '32k/validation.jsonl': {'sha256': 'c40c876828c0338036fdde844e1d47f7d66f575dfc22e5e9721a33dc39489d74', 'bytes': 4696845}, 'wheels/accelerate-1.15.0-py3-none-any.whl': {'sha256': '97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec', 'bytes': 394295}, 'wheels/fla_core-0.6.0-py3-none-any.whl': {'sha256': 'b0708b70e0b119cd58a0a5146ea56cb175cef6ed969d8ac60267e3e077ce3ebd', 'bytes': 1085221}, 'wheels/flash_linear_attention-0.6.0-py3-none-any.whl': {'sha256': '6dec9735f8454523c13553622e98aff30152dde6586b4ac7d88010fbb7a06b5f', 'bytes': 403549}, 'wheels/peft-0.21.2-py3-none-any.whl': {'sha256': '106ab6077ff72c54d21577f9af5970e34bac7582cb14209f7b2b511e322a4eae', 'bytes': 835427}, 'arc3_training_collator.py': {'sha256': 'e27f3881e0b83ab3f3cdc4da216158ed5dd69e6807891e5ea925702766170858', 'bytes': 2460}, 'split.json': {'sha256': 'ce36006c553278e8282b9e8ca05bfa0b6f6b39a9ffc831d03dd9419e760e5525', 'bytes': 1233}, 'base-requirements.lock': {'sha256': '4b0ee12848c1e7c3a51b0819de3a1ae63a967313fd1f62b9e2ca6b9c82b8da02', 'bytes': 4187}}, 'purpose': 'Private offline ARC3 training/evaluation inputs. No pretrained weights, credentials, hidden games or test-game examples.', 'attribution': 'Agent/prompt source Daniel Franzen, Jeroen Cottaar and Tufa Labs, Apache2.0. Our own recorded Qwen agent outputs. Upstream dependency licenses remain inside wheels. Preserve applicable original notices; no blanket relicensing of third-party material.'}
MODEL_PATH='/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1'
BF16_PATH='/kaggle/input/models/dangkhoa2016/qwen-qwen3-8-27b/transformers/default/1'
EXPECTED_TARGET_METADATA={'config.json': {'sha256': '74227dd615bf1ea975aa676bdf355a0379858c12f394b5365cd9dfa5fc2c70bc', 'size': 51350}, 'tokenizer.json': {'sha256': '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3', 'size': 12809320}, 'tokenizer_config.json': {'sha256': 'b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27', 'size': 17928}, 'chat_template.jinja': {'sha256': 'c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041', 'size': 8952}, 'preprocessor_config.json': {'sha256': '27225450ac9c6529872ee1924fcb0962ff5634834f817040f444118116f4e516', 'size': 390}}
from hashlib import sha256
work=WORKING_DIR
bundle=WORKING_DIR/'training-dependencies'
bundle.mkdir(exist_ok=True)
import tarfile
input_roots=[Path('/kaggle/input/datasets/kunalsinghh25/arc3-training-pilot-inputs-v1'),Path('/kaggle/input/arc3-training-pilot-inputs-v1')]
assets=WORKING_DIR/'training-assets';assets.mkdir(exist_ok=True)
archives=[root/'training-pilot-inputs.tar.gz' for root in input_roots if (root/'training-pilot-inputs.tar.gz').is_file()]
if archives:
    assert len(archives)==1
    archive=archives[0]
    assert sha256(archive.read_bytes()).hexdigest()==ARCHIVE_SHA256,'Private input archive changed'
    tar=tarfile.open(archive,'r:gz')
    manifest=json.loads(tar.extractfile('manifest.json').read())
    def read_asset(name):
        member=tar.getmember(name)
        if not member.isfile():raise ValueError('Invalid archive member')
        return tar.extractfile(member).read()
    input_asset_mode='archive-sha256-and-file-sha256'
else:
    # Kaggle automatically expanded this uploaded tar.gz. The API file listing
    # verifies this exact prefix; validate every original manifest/file hash.
    roots=[root/'training-pilot-inputs' for root in input_roots if (root/'training-pilot-inputs/manifest.json').is_file()]
    if len(roots)!=1:raise ValueError('Pinned expanded assets are not uniquely mounted: '+str(roots))
    source_root=roots[0].resolve();manifest=json.loads((source_root/'manifest.json').read_text())
    def read_asset(name):
        source=(source_root/name).resolve()
        if not source.is_relative_to(source_root) or not source.is_file():raise ValueError('Invalid expanded asset path')
        return source.read_bytes()
    input_asset_mode='kaggle-expanded-exact-manifest-and-file-sha256'
assert manifest==EXPECTED_INPUT_MANIFEST
try:
    for name,record in manifest['files'].items():
        destination=(assets/name).resolve()
        if not destination.is_relative_to(assets.resolve()):raise ValueError('Asset escapes output root')
        content=read_asset(name)
        assert len(content)==record['bytes'] and sha256(content).hexdigest()==record['sha256'],name
        destination.parent.mkdir(parents=True,exist_ok=True);destination.write_bytes(content)
finally:
    if archives:tar.close()
(assets/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('TRAINING_ASSET_INTEGRITY',input_asset_mode,flush=True)
model=Path(MODEL_PATH);bf16=Path(BF16_PATH)
assert sha256((bf16/'tokenizer.json').read_bytes()).hexdigest()==EXPECTED_TARGET_METADATA['tokenizer.json']['sha256'],'BF16 training tokenizer differs from deployment target'
bc=json.loads((bf16/'config.json').read_text());tc=json.loads((model/'config.json').read_text())
for key in ['vocab_size','hidden_size','num_hidden_layers','num_attention_heads','num_key_value_heads']:
    assert bc['text_config'][key]==tc['text_config'][key],key
assert bc['architectures']==tc['architectures']==['Qwen3_5ForConditionalGeneration']
for wheel in (assets/'wheels').glob('*.whl'):(bundle/wheel.name).symlink_to(wheel)
run([*uv,'pip','install','--python',PYTHON,'--no-index','--no-deps',*sorted(bundle.glob('*.whl'))],env=install_env)
run([*uv,'pip','check','--python',PYTHON],env=install_env)
(WORKING_DIR/'arc3_lora_trainer.py').write_text('"""Bounded actual-checkpoint ARC3 teacher adaptation. Imported only in training jobs."""\nimport argparse\nfrom collections import defaultdict\nfrom hashlib import sha256\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport time\n\nSUPPORTED_PROJECTIONS = frozenset([\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\',\n    \'in_proj_qkv\',\'in_proj_z\',\'out_proj\',\'gate_proj\',\'up_proj\',\'down_proj\'])\n\n\ndef select_lora_targets(linear_names):\n    """Native language projections supported by the pinned serving runtime.\n\n    Exclude vision, output embeddings and GDN beta/decay projections. The latter\n    are not in the deployed runtime\'s supported packed adapter families.\n    """\n    return sorted(name for name in linear_names if \'language_model.layers.\' in name\n        and name.rsplit(\'.\',1)[-1] in SUPPORTED_PROJECTIONS)\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--assets\',type=Path,required=True)\n    parser.add_argument(\'--model\',type=Path,required=True)\n    parser.add_argument(\'--processor\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--deadline\',type=float,required=True,help=\'Absolute epoch time, before the notebook finalization reserve\')\n    args=parser.parse_args()\n    import torch\n    import torch.nn.functional as functional\n    from peft import LoraConfig,get_peft_model\n    from transformers import AutoModelForImageTextToText,AutoProcessor\n    from transformers.models.qwen3_5.modeling_qwen3_5 import chunk_gated_delta_rule\n    if not torch.cuda.is_available():raise RuntimeError(\'Actual CUDA device required\')\n    if not callable(chunk_gated_delta_rule):raise RuntimeError(\'Fast differentiable GDN kernels are unavailable\')\n    torch.set_num_threads(8);torch.manual_seed(20261101);random.seed(20261101)\n    output=args.output;output.mkdir(parents=True,exist_ok=True)\n    report_path=output/\'training-report.json\'\n    if (output/\'adapter\').exists():raise RuntimeError(\'Refusing to overwrite an existing adapter\')\n    started=time.time()\n    report=dict(phase=\'loading\',started_at_epoch=started,deadline=args.deadline,\n        trainable_scope=\'Rank16, alpha32,400 supported language attention/recurrent/feed-forward projections; frozen base and vision. GDN beta/decay and embeddings excluded.\',\n        seed=20261101,learning_rate=1e-5,gradient_accumulation=4,max_epochs=1,\n        device=torch.cuda.get_device_name(0),gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory,\n        optimizer_steps=0,training_examples=0,training_input_tokens=0,supervised_tokens=0,\n        examples=[],validation=[],boundary=\'Actual teacher-imitation training, not a gameplay or official score. Validation loss and serialization checks do not establish unfamiliar-game improvement.\')\n    def save():\n        report[\'updated_at_epoch\']=time.time()\n        tmp=report_path.with_suffix(\'.tmp\');tmp.write_text(json.dumps(report,indent=2)+\'\\n\');tmp.replace(report_path)\n    save()\n    try:\n        from arc3_lora_export import preflight_export\n        report[\'export_preflight\']=preflight_export(output/\'export-preflight\',Path(__file__).with_name(\'sglang_gdn_normalizer.py\').read_text())\n        save()\n        manifest=json.loads((args.assets/\'manifest.json\').read_text())\n        for name,record in manifest[\'files\'].items():\n            if sha256((args.assets/name).read_bytes()).hexdigest()!=record[\'sha256\']:\n                raise ValueError(\'Training asset changed: \'+name)\n        report[\'dataset_sha256\']={s:manifest[\'files\'][\'16k/\'+s+\'.jsonl\'][\'sha256\'] for s in [\'train\',\'validation\']}\n        train=[json.loads(line) for line in (args.assets/\'16k/train.jsonl\').read_text().splitlines()]\n        validation=[json.loads(line) for line in (args.assets/\'16k/validation.jsonl\').read_text().splitlines()]\n        split=json.loads((args.assets/\'split.json\').read_text())[\'games\']\n        if any(split[r[\'metadata\'][\'game_id\']]!=\'train\' for r in train):raise ValueError(\'Training game split violation\')\n        if any(split[r[\'metadata\'][\'game_id\']]!=\'validation\' for r in validation):raise ValueError(\'Validation game split violation\')\n        # A fixed two-example sample per validation game, selected without student losses.\n        grouped=defaultdict(list)\n        for row in validation:grouped[row[\'metadata\'][\'game_id\']].append(row)\n        validation_sample=[]\n        for game,rows in sorted(grouped.items()):\n            validation_sample.extend(sorted(rows,key=lambda r:sha256((\'arc3-fixed-val-v1/\'+r[\'example_id\']).encode()).hexdigest())[:2])\n        report[\'validation_sample_ids\']=[r[\'example_id\'] for r in validation_sample]\n        namespace={};exec((args.assets/\'arc3_training_collator.py\').read_text(),namespace)\n        collate=namespace[\'collate_example\']\n        processor=AutoProcessor.from_pretrained(args.processor,local_files_only=True,trust_remote_code=False)\n        native=AutoModelForImageTextToText.from_pretrained(args.model,torch_dtype=torch.bfloat16,\n            device_map={\'\':\'cuda:0\'},attn_implementation=\'sdpa\',local_files_only=True,trust_remote_code=False)\n        native.config.use_cache=False\n        targets=select_lora_targets(name for name,module in native.named_modules() if isinstance(module,torch.nn.Linear))\n        if len(targets)!=400:raise ValueError(\'Unexpected supported language adapter placement\')\n        model=get_peft_model(native,LoraConfig(r=16,lora_alpha=32,target_modules=targets,lora_dropout=0.0,\n            bias=\'none\',task_type=\'CAUSAL_LM\'))\n        parameters=[p for p in model.parameters() if p.requires_grad]\n        trainable=[name for name,p in model.named_parameters() if p.requires_grad]\n        if not parameters or any(\'lora_\' not in name for name in trainable):raise ValueError(\'Frozen-base invariant violated\')\n        if sum(p.numel() for p in parameters)!=108789760:raise ValueError(\'Native adapter shapes differ from the executed CPU module inventory\')\n        report.update(phase=\'baseline_validation\',trainable_parameters=sum(p.numel() for p in parameters),lora_targets=targets,\n            model_loaded_at_epoch=time.time(),training_examples_available=len(train))\n        save()\n        # Verify selected-position CE against the model\'s built-in shifted loss\n        # on a short actual-checkpoint forward before spending time on training.\n        model.eval()\n        calibration=processor.tokenizer(\'Observe the grid, test a hypothesis, compare the outcome and revise the model.\',return_tensors=\'pt\',add_special_tokens=False)\n        calibration={key:value.to(\'cuda:0\') for key,value in calibration.items()}\n        calibration_labels=calibration[\'input_ids\'].clone();calibration_labels[:,:-6]=-100\n        calibration_positions=torch.nonzero(calibration_labels[0]!=-100,as_tuple=False).flatten()\n        with torch.no_grad(),torch.autocast(\'cuda\',dtype=torch.bfloat16):\n            builtin=model(**calibration,labels=calibration_labels,use_cache=False).loss\n            chosen=model(**calibration,use_cache=False,logits_to_keep=calibration_positions-1).logits[0]\n            selected=functional.cross_entropy(chosen.float(),calibration_labels[0,calibration_positions])\n        alignment_error=abs(float(builtin)-float(selected))\n        if not math.isfinite(alignment_error) or alignment_error>1e-3:raise ValueError(\'Selected next-token loss differs from native shifted loss\')\n        report[\'native_loss_alignment\']=dict(native_loss=float(builtin),selected_loss=float(selected),absolute_error=alignment_error,\n            boundary=\'Short actual-checkpoint mathematical loss alignment check, not an ARC3 capability evaluation.\')\n        del calibration,calibration_labels,calibration_positions,builtin,chosen,selected\n        save()\n        def loss_for(row):\n            batch,info=collate(row,processor,max_tokens=16384)\n            batch={key:value.to(\'cuda:0\') for key,value in batch.items()}\n            labels=batch.pop(\'labels\')\n            positions=torch.nonzero(labels[0]!=-100,as_tuple=False).flatten()\n            if positions.numel()<16 or int(positions.min())<1:raise ValueError(\'Invalid next-token supervision positions\')\n            # Compute vocabulary logits only at supervised positions. This is\n            # exactly masked next-token CE, avoiding a16K *248K full logits tensor.\n            with torch.autocast(\'cuda\',dtype=torch.bfloat16):\n                result=model(**batch,use_cache=False,logits_to_keep=positions-1)\n                logits=result.logits[0]\n            expected=labels[0,positions]\n            if logits.shape[0]!=expected.numel():raise ValueError(\'Selected logits/labels mismatch\')\n            loss=functional.cross_entropy(logits.float(),expected,reduction=\'mean\')\n            return loss,info\n        def validate(tag):\n            model.eval();weighted=tokens=0;values=[]\n            with torch.no_grad():\n                for row in validation_sample:\n                    loss,info=loss_for(row)\n                    if not torch.isfinite(loss):raise FloatingPointError(\'Non-finite validation loss\')\n                    n=info[\'supervised_tokens\'];weighted+=float(loss)*n;tokens+=n\n                    values.append(dict(example=row[\'example_id\'],loss=float(loss),supervised_tokens=n))\n            value=dict(tag=tag,mean_token_loss=weighted/tokens,supervised_tokens=tokens,examples=values)\n            report[\'validation\'].append(value);save();return value\n        before=validate(\'before_training\')\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n        model.enable_input_require_grads()\n        optimizer=torch.optim.AdamW(parameters,lr=1e-5,betas=(0.9,0.999),weight_decay=0.0)\n        order=list(range(len(train)));random.Random(20261101).shuffle(order)\n        report.update(phase=\'training\',training_started_at_epoch=time.time());save()\n        durations=[]\n        for group_start in range(0,len(order),4):\n            # Reserve for at least one more observed-duration group; external\n            # notebook timeout still independently bounds a stalled CUDA call.\n            estimate=max(durations[-4:],default=60.0)\n            if time.time()+estimate*min(4,len(order)-group_start)+30>=args.deadline:break\n            model.train();native.model.visual.eval()\n            optimizer.zero_grad(set_to_none=True)\n            indices=order[group_start:group_start+4];processed=0\n            for index in indices:\n                if processed and time.time()+max(durations[-4:],default=60.0)+15>=args.deadline:break\n                tick=time.time();loss,info=loss_for(train[index])\n                if not torch.isfinite(loss):raise FloatingPointError(\'Non-finite real-checkpoint training loss\')\n                (loss/len(indices)).backward();torch.cuda.synchronize()\n                duration=time.time()-tick;durations.append(duration);processed+=1\n                report[\'training_examples\']+=1;report[\'training_input_tokens\']+=info[\'actual_input_tokens\']\n                report[\'supervised_tokens\']+=info[\'supervised_tokens\']\n                report[\'examples\'].append(dict(example=train[index][\'example_id\'],loss=float(loss.detach()),wall_seconds=duration,**info))\n                save()\n            if not processed:break\n            if processed<len(indices):\n                for parameter in parameters:\n                    if parameter.grad is not None:parameter.grad.mul_(len(indices)/processed)\n            gradients=[p.grad for p in parameters if p.grad is not None]\n            if not gradients or not all(bool(torch.isfinite(g).all()) for g in gradients):raise FloatingPointError(\'Missing/non-finite actual gradients\')\n            norm=torch.nn.utils.clip_grad_norm_(parameters,1.0)\n            if not torch.isfinite(norm) or float(norm)<=0:raise FloatingPointError(\'No finite nonzero adapter update\')\n            optimizer.step();report[\'optimizer_steps\']+=1\n            if report[\'optimizer_steps\']==1 or report[\'optimizer_steps\']%8==0:\n                model.save_pretrained(output/\'checkpoints\'/(\'step-\'+str(report[\'optimizer_steps\'])),safe_serialization=True)\n            report[\'last_gradient_norm\']=float(norm)\n            report[\'max_gpu_allocated_bytes\']=torch.cuda.max_memory_allocated()\n            report[\'max_gpu_reserved_bytes\']=torch.cuda.max_memory_reserved()\n            print(json.dumps(dict(optimizer_step=report[\'optimizer_steps\'],examples=report[\'training_examples\'],\n                last_loss=report[\'examples\'][-1][\'loss\'],gradient_norm=float(norm))),flush=True)\n            save()\n        if report[\'optimizer_steps\']<1:raise RuntimeError(\'No actual optimizer update completed within the training budget\')\n        optimizer.zero_grad(set_to_none=True)\n        report.update(phase=\'checkpoint_validation\',training_finished_at_epoch=time.time());save()\n        after=validate(\'after_training\')\n        adapter=output/\'adapter\';model.save_pretrained(adapter,safe_serialization=True)\n        nonzero=sum(float(parameter.detach().float().square().sum()) for name,parameter in model.named_parameters() if \'lora_B\' in name)\n        if not math.isfinite(nonzero) or nonzero<=0:raise ValueError(\'Saved adapter has no nonzero output update\')\n        report[\'adapter_output_squared_norm\']=nonzero\n        from arc3_lora_export import export_serving_adapter\n        report[\'serving_export\']=export_serving_adapter(adapter,output/\'serving-adapter\',native.config.text_config.to_dict())\n        save()\n        # Prove the saved adapter loads into a separate adapter slot and is used;\n        # zero the original slot after loading so it cannot mask a broken reload.\n        model.load_adapter(adapter,adapter_name=\'reloaded\',is_trainable=False)\n        model.set_adapter(\'reloaded\')\n        with torch.no_grad():\n            for name,parameter in model.named_parameters():\n                if \'lora_\' in name and \'.default.\' in name:parameter.zero_()\n        reloaded=validate(\'saved_adapter_reloaded\')\n        error=abs(reloaded[\'mean_token_loss\']-after[\'mean_token_loss\'])\n        if error>1e-3:raise ValueError(\'Saved-adapter reload does not reproduce loss: \'+str(error))\n        report.update(phase=\'complete\',adapter_files={p.name:dict(bytes=p.stat().st_size,sha256=sha256(p.read_bytes()).hexdigest()) for p in adapter.iterdir() if p.is_file()},\n            reloaded_adapter_active=str(model.active_adapter),reload_loss_absolute_error=error,\n            validation_loss_delta=after[\'mean_token_loss\']-before[\'mean_token_loss\'],\n            training_epoch_fraction=report[\'training_examples\']/len(train),\n            finished_at_epoch=time.time(),serving_verified=False,gameplay_improvement_verified=False,official_submission_eligible=False)\n        save()\n    except BaseException as exc:\n        report.update(phase=\'failed\',error_type=type(exc).__name__,error=str(exc)[:800]);save();raise\n\n\nif __name__==\'__main__\':main()\n')
(WORKING_DIR/'arc3_lora_export.py').write_text('"""Export native Qwen3.5 GDN adapters to the pinned SGLang split format."""\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\n\nPACKED_TARGETS=[\'qkv_proj\',\'o_proj\',\'out_proj\',\'in_proj_qkvz\',\'gate_up_proj\',\'down_proj\']\n\n\ndef preflight_export(output,normalizer_source):\n    """Execute the pinned runtime\'s normalizer on an actual tiny export."""\n    import torch\n    from safetensors.torch import load_file,save_file\n    output=Path(output);native=output/\'native\';native.mkdir(parents=True)\n    generator=torch.Generator().manual_seed(20261101)\n    weights={}\n    for layer in range(48):\n        for projection,width in [(\'in_proj_qkv\',8),(\'in_proj_z\',4)]:\n            prefix=f\'base_model.model.model.language_model.layers.{layer}.linear_attn.{projection}\'\n            weights[prefix+\'.lora_A.weight\']=torch.randn(2,6,generator=generator)\n            weights[prefix+\'.lora_B.weight\']=torch.randn(width,2,generator=generator)\n    save_file(weights,str(native/\'adapter_model.safetensors\'))\n    (native/\'adapter_config.json\').write_text(json.dumps(dict(r=2,lora_alpha=4,target_modules=[\'in_proj_qkv\',\'in_proj_z\'])))\n    config=dict(linear_key_head_dim=1,linear_num_key_heads=2,linear_value_head_dim=1,linear_num_value_heads=4)\n    exported=export_serving_adapter(native,output/\'serving\',config)\n    converted=load_file(str(output/\'serving/adapter_model.safetensors\'))\n    namespace={};exec(normalizer_source,namespace)\n    namespace[\'_normalize_in_proj_qkvz\'](None,converted)\n    probe=torch.randn(6,3,generator=generator);errors=[]\n    for layer in range(48):\n        prefix=f\'base_model.model.model.language_model.layers.{layer}.linear_attn.\'\n        expected=torch.cat([weights[prefix+p+\'.lora_B.weight\']@(weights[prefix+p+\'.lora_A.weight\']@probe)\n            for p in [\'in_proj_qkv\',\'in_proj_z\']],dim=0)\n        a=converted[prefix+\'in_proj_qkvz.lora_A.weight\'];b=converted[prefix+\'in_proj_qkvz.lora_B.weight\']\n        actual=torch.cat([bb@(aa@probe) for aa,bb in zip(a.split(2,dim=0),b.split([2,2,4,4],dim=0))],dim=0)\n        if not torch.allclose(actual,expected,rtol=1e-4,atol=1e-5):raise ValueError(\'Pinned serving normalizer changes native recurrent adapter math\')\n        errors.append(float((actual-expected).abs().max()))\n    return dict(passed=True,actual_runtime_normalizer_sha256=sha256(normalizer_source.encode()).hexdigest(),\n        groups_checked=len(errors),max_absolute_error=max(errors),boundary=\'Actual Torch/safetensors export and pinned runtime normalization on tiny synthetic tensors only; saved trained weights and gameplay remain required.\')\n\n\ndef export_serving_adapter(native_adapter,output,text_config):\n    import torch\n    from safetensors.torch import load_file,save_file\n    native_adapter=Path(native_adapter);output=Path(output)\n    if output.exists():raise ValueError(\'Refusing to overwrite serving export\')\n    weights=load_file(str(native_adapter/\'adapter_model.safetensors\'),device=\'cpu\')\n    config=json.loads((native_adapter/\'adapter_config.json\').read_text())\n    key=int(text_config[\'linear_key_head_dim\'])*int(text_config[\'linear_num_key_heads\'])\n    value=int(text_config[\'linear_value_head_dim\'])*int(text_config[\'linear_num_value_heads\'])\n    if config.get(\'rank_pattern\') or config.get(\'alpha_pattern\'):\n        raise ValueError(\'Only uniform rank/scaling is supported\')\n    converted={};checks=[]\n    for name in sorted(weights):\n        tensor=weights[name]\n        if \'.in_proj_qkv.\' not in name:\n            converted[name]=tensor\n            continue\n        if \'.lora_A.\' in name:\n            for suffix in [\'q\',\'k\',\'v\']:\n                converted[name.replace(\'.in_proj_qkv.\',\'.in_proj_\'+suffix+\'.\')]=tensor.clone()\n        elif \'.lora_B.\' in name:\n            if tensor.shape[0]!=2*key+value:raise ValueError(\'Unexpected QKV adapter output width\')\n            for suffix,part in zip([\'q\',\'k\',\'v\'],tensor.split([key,key,value],dim=0)):\n                converted[name.replace(\'.in_proj_qkv.\',\'.in_proj_\'+suffix+\'.\')]=part.contiguous()\n            a_name=name.replace(\'.lora_B.\',\'.lora_A.\')\n            a=weights[a_name].float()\n            # Check the actual trained delta on deterministic probe vectors;\n            # no gigantic dense B@A matrix is materialized.\n            probe=torch.linspace(-1,1,a.shape[1]*2,dtype=torch.float32).reshape(a.shape[1],2)\n            expected=tensor.float()@(a@probe)\n            actual=torch.cat([converted[name.replace(\'.in_proj_qkv.\',\'.in_proj_\'+s+\'.\')].float()\n                @(converted[a_name.replace(\'.in_proj_qkv.\',\'.in_proj_\'+s+\'.\')].float()@probe)\n                for s in [\'q\',\'k\',\'v\']],dim=0)\n            error=float((actual-expected).abs().max())\n            if not torch.allclose(actual,expected,rtol=1e-4,atol=1e-5):\n                raise ValueError(\'Serving QKV export changes the trained delta\')\n            z_name=name.replace(\'.in_proj_qkv.\',\'.in_proj_z.\')\n            if z_name not in weights or weights[z_name].shape[0]!=value:\n                raise ValueError(\'Matching native Z adapter missing\')\n            checks.append(dict(native=name,max_absolute_error=error,widths=[key,key,value]))\n        else:raise ValueError(\'Unrecognized native QKV adapter tensor\')\n    if len(checks)!=48:raise ValueError(\'Expected48 native recurrent QKV adapters\')\n    config[\'target_modules\']=PACKED_TARGETS\n    config[\'inference_mode\']=True\n    output.mkdir(parents=True)\n    save_file({name:tensor.contiguous() for name,tensor in converted.items()},str(output/\'adapter_model.safetensors\'),metadata={\'format\':\'pt\'})\n    (output/\'adapter_config.json\').write_text(json.dumps(config,indent=2)+\'\\n\')\n    report=dict(format=\'SGLang d00d88efc8d6 GDN four-slice adapter\',native_tensor_count=len(weights),\n        serving_tensor_count=len(converted),qkv_delta_checks=checks,target_modules=PACKED_TARGETS,\n        serving_verified=False,boundary=\'Actual-weight algebraic export only. Runtime load, routing and gameplay remain required.\')\n    (output/\'export-report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    report[\'files\']={p.name:dict(bytes=p.stat().st_size,sha256=sha256(p.read_bytes()).hexdigest()) for p in output.iterdir() if p.is_file()}\n    return report\n')
(WORKING_DIR/'sglang_gdn_normalizer.py').write_text('import torch\nfrom typing import Dict\ndef _normalize_in_proj_qkvz(self, weights: Dict[str, torch.Tensor]):\n        """Normalize in_proj_qkvz weights for GDN (GatedDeltaNet) layers like\n        Qwen3.5.\n\n        Two adapter formats are handled:\n\n        1. Split: ``in_proj_q + in_proj_k + in_proj_v + in_proj_z`` are present\n           as separate weights → concatenate them into ``in_proj_qkvz``.\n\n        2. Already-merged: the adapter has a single ``in_proj_qkvz`` weight\n           (PEFT trained against SGLang\'s fused Linear). The stacked buffer\n           expects four per-slice ``A`` blocks, so repeat ``lora_A`` 4× along\n           the rank dim. ``lora_B`` is already full-output-dim and matches\n           the buffer directly.\n        """\n        for weight_name in list(weights.keys()):\n            if "in_proj_q." in weight_name:\n                k_name = weight_name.replace("in_proj_q", "in_proj_k")\n                v_name = weight_name.replace("in_proj_q", "in_proj_v")\n                z_name = weight_name.replace("in_proj_q", "in_proj_z")\n                if (\n                    k_name not in weights\n                    or v_name not in weights\n                    or z_name not in weights\n                ):\n                    continue\n                qkvz_name = weight_name.replace("in_proj_q", "in_proj_qkvz")\n                cat_dim = weights[weight_name].dim() - 2\n                weights[qkvz_name] = torch.cat(\n                    (\n                        weights[weight_name],\n                        weights[k_name],\n                        weights[v_name],\n                        weights[z_name],\n                    ),\n                    cat_dim,\n                )\n                weights.pop(weight_name)\n                weights.pop(k_name)\n                weights.pop(v_name)\n                weights.pop(z_name)\n            elif "in_proj_qkvz" in weight_name and "lora_A" in weight_name:\n                # Already-merged adapter: replicate the shared A across the 4\n                # stacked slots the buffer expects (q, k, v, z).\n                ndim = weights[weight_name].dim()\n                repeat_dims = [1] * ndim\n                repeat_dims[ndim - 2] = 4\n                weights[weight_name] = weights[weight_name].repeat(*repeat_dims)\n')
training_deadline=NOTEBOOK_START_TIME+7200-900
if time.time()+1200>=training_deadline:raise RuntimeError('Startup left too little actual training time')
training_output=WORKING_DIR/'sft-pilot'
command=[PYTHON,str(WORKING_DIR/'arc3_lora_trainer.py'),'--assets',str(assets),'--model',MODEL_DIR,
    '--processor',MODEL_PATH,'--output',str(training_output),'--deadline',str(training_deadline)]
log=WORKING_DIR/'training.log'
with log.open('w') as stream:
    process=subprocess.Popen(command,env=env,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
    try:
        while process.poll() is None:
            status_path=training_output/'training-report.json'
            status=json.loads(status_path.read_text()) if status_path.is_file() else {}
            if time.time()>NOTEBOOK_START_TIME+2400 and status.get('optimizer_steps',0)<1:
                raise RuntimeError('No real optimizer step within40minutes including startup; stop the owned training process')
            if time.time()>NOTEBOOK_START_TIME+7200-120:
                raise RuntimeError('Owned training process exceeded finalization reserve')
            time.sleep(10)
        if process.returncode:raise RuntimeError('Actual training child failed; inspect training.log/report')
    finally:
        if process.poll() is None:
            os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=20)
            except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait(timeout=10)
report=json.loads((training_output/'training-report.json').read_text())
if report['phase']!='complete' or report['optimizer_steps']<1:raise RuntimeError('No complete actual training result')
print(json.dumps({k:v for k,v in report.items() if k not in ['examples','validation','lora_targets']},indent=2))
